In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_SZ = 0.8
VALID_SZ = 1 - TRAIN_SZ

# Read the data
data = pd.read_csv('../datasets/melbourne-housing-snapshot/melb_data.csv')

# Separate target from predictors
y = data.Price
X = data.drop(['Price'], axis=1)

# Divide data into training and validation subsets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, 
                                                      train_size=TRAIN_SZ, test_size=VALID_SZ,
                                                      random_state=0)

# Drop columns with missing values (simplest approach)
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()] 
X_train.drop(cols_with_missing, axis=1, inplace=True)
X_valid.drop(cols_with_missing, axis=1, inplace=True)

# Select categorical columns with relatively low cardinality (convenient but arbitrary)
low_cardinality_cols = [col for col in X_train.columns 
                        if X_train[col].dtype == "object"
                        and X_train[col].nunique() < 10]

# Select numerical columns
numerical_cols = [col for col in X_train.columns 
                  if X_train[col].dtype in ['int64', 'float64']]

# Keep selected columns only
predictor_cols = low_cardinality_cols + numerical_cols
X_train = X_train[predictor_cols]
X_valid = X_valid[predictor_cols]

In [2]:
X_train.head()

,Type,Method,Regionname,Rooms,Distance,Postcode,Bedroom2,Bathroom,Landsize,Lattitude,Longtitude,Propertycount
12167,u,S,Southern Metropolitan,1,5.0,3182.0,1.0,1.0,0.0,-37.85984,144.9867,13240.0
6524,h,SA,Western Metropolitan,2,8.0,3016.0,2.0,2.0,193.0,-37.85800,144.9005,6380.0
8413,h,S,Western Metropolitan,3,12.6,3020.0,3.0,1.0,555.0,-37.79880,144.8220,3755.0
2919,u,SP,Northern Metropolitan,3,13.0,3046.0,3.0,1.0,265.0,-37.70830,144.9158,8870.0
6043,h,S,Western Metropolitan,3,13.3,3020.0,3.0,1.0,673.0,-37.76230,144.8272,4217.0


In [3]:
X_valid.head()

,Type,Method,Regionname,Rooms,Distance,Postcode,Bedroom2,Bathroom,Landsize,Lattitude,Longtitude,Propertycount
8505,h,SP,Western Metropolitan,4,8.0,3016.0,4.0,2.0,450.0,-37.86100,144.89850,6380.0
5523,h,S,Western Metropolitan,2,6.6,3011.0,2.0,1.0,172.0,-37.81000,144.88960,2417.0
12852,h,SP,Western Metropolitan,3,10.5,3020.0,3.0,1.0,581.0,-37.76740,144.82421,4217.0
4818,t,PI,Southern Metropolitan,3,4.5,3181.0,2.0,2.0,128.0,-37.85260,145.00710,7717.0
12812,h,S,Northern Metropolitan,3,8.5,3044.0,3.0,2.0,480.0,-37.72523,144.94567,7485.0


In [4]:
# Get list of categorical variables
s = (X_train.dtypes == 'object')
object_cols = list(s[s].index)
print(f"Categorical variables: {object_cols}")

Categorical variables: ['Type', 'Method', 'Regionname']


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [6]:
### Approach #1: Drop Categorical Variables
drop_X_train = X_train.copy().select_dtypes(exclude=['object'])
drop_X_valid = X_valid.copy().select_dtypes(exclude=['object'])

drop_mae = score_dataset(drop_X_train, drop_X_valid, y_train, y_valid)
print(f"MAE from Approach 1 (Drop categorical variables): {drop_mae}")

MAE from Approach 1 (Drop categorical variables): 175703.48185157913


In [7]:
### Approach #2: Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder

# Make copy to avoid changing original data 
label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

# Apply ordinal encoder to each column with categorical data
ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])

label_mae = score_dataset(label_X_train, label_X_valid, y_train, y_valid)
print(f"MAE from Approach 2 (Ordinal Encoding): {label_mae}") 

MAE from Approach 2 (Ordinal Encoding): 165936.40548390493


In [10]:
### Approach #3: One-Hot Encoding
from sklearn.preprocessing import OneHotEncoder

# Apply one-hot encoder to each column with categorical data
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.copy().drop(object_cols, axis=1)
num_X_valid = X_valid.copy().drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

oh_mae = score_dataset(OH_X_train, OH_X_valid, y_train, y_valid)
print(f"MAE from Approach 3 (One-Hot Encoding): {oh_mae}") 


MAE from Approach 3 (One-Hot Encoding): 166089.4893009678
